# Seq-cell constraint LUT — single-task visualization (fine_x10_loopclose)

Mirror the **validation script (`TSMC_GCN_topology_validation.py`) + `run_predictions_with_adaptation`** end-to-end for ONE random task, so we can eyeball:
- the V-direction smoothness of the constraint LUT (is the lib characterization itself non-monotone?),
- where the predicted 61-V curve diverges from the actual (which voltages the model can't fit),
- whether the v6 per-task sign_flip is firing.

Aligned with the lib-generation pipeline (per the parity-check memory note):
1. `output_load → input_slew` norm alias for constraint mode
2. center voltage = `(0.9 - voltage_mean) / voltage_std` projected onto voltage-bearing nodes (mask `!= 0`)
3. Per-task **v6 sign_flip**: `y ← -y` when `y_high > y_low`, unflipped at the end (pure mirror, no abs)
4. Temperature normalization uses MOS mask (column 2 != 0) — matches train/validation/lib-gen
5. **selective_adam**: Adam (lr=3e-4, wd=1e-4, 40 steps) only if support-set MSE > 1e-4
6. Interpolation indices [0, 13, 30, 45, 60]

Dataset: `GNN_dataset_TSMC_fine_x10_loopclose`. Cache: `..._loopclose.pth`.

In [ ]:
import os, sys, copy, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch_geometric.data import Data, Batch

ROOT = '/home/tkdgn2907/Deepsets_test/MAML/Projects'
for p in [
    f'{ROOT}/model_code',
    f'{ROOT}/data_processing/gnn',
    f'{ROOT}/pretraining/model_test_code/utils',
    f'{ROOT}/pretraining/model_test_code/gnn',
    f'{ROOT}/Lib_file_generation',
]:
    if p not in sys.path:
        sys.path.insert(0, p)

from gnn_maml import create_maml_gcn_model
from gnn_functions import model_functions_at_training_gnn
# Validation's CellTestDataset has get_all_libs_for_task + per-sample
# delay_type/output_name metadata (lib-gen's variant lacks get_all_libs_for_task).
from TSMC_GCN_topology_validation import normalize_node_features, CellTestDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device = {device}')

## 1. Pick a task

In [ ]:
# ----- configurable -----
CELL_NAME    = 'DFCNQD1BWP30P140'
CATEGORY     = 'setup'     # 'setup'|'hold'|'recovery'|'removal'|'non_seq_setup'|'non_seq_hold'|'cell'|'transition'
SUPPORT_IDX  = [0, 13, 30, 45, 60]   # interpolation 5-shot (matches validation default)
SEED         = 42
# ------------------------

IS_CONSTRAINT = CATEGORY in ('setup', 'hold', 'recovery', 'removal', 'non_seq_setup', 'non_seq_hold')

DATASET_DIR  = f'{ROOT}/dataset_all/GNN_dataset_TSMC_fine_x10_loopclose'
CACHE_PATH   = f'{ROOT}/data_processing/gnn/topology_cache/stage_aware_topology_cache_tsmc_tcbn28hpcplusbwp30p140_110a_lpe_typical_loopclose.pth'
CKPT_PATH    = f'{ROOT}/pretrained_models/gnn_maml_tsmc_process_checkpoints/gnn_maml_tsmc_process_cell_stage_aware_innerdiv10_meta16_iter300000_inner1_conv64x2_fc256x2.pth'

test_pth = f'{DATASET_DIR}/test_by_{CATEGORY}_stage_aware/{CELL_NAME}.pth'

print('Loading topology cache...')
topology_cache = torch.load(CACHE_PATH, weights_only=False, map_location='cpu', mmap=True)
print(f'  topology cells: {len(topology_cache)}')

print(f'Loading dataset: {os.path.basename(os.path.dirname(test_pth))}/{os.path.basename(test_pth)}')
dataset = CellTestDataset(test_pth)
print(f'  num_tasks={dataset.num_tasks}, num_libs={dataset.num_libs}')

rng = random.Random(SEED)
task_idx = rng.sample(range(dataset.num_tasks), 1)[0]
task_samples, task_outputs_list = dataset.get_all_libs_for_task(task_idx, clone=True)
y61_signed = np.array([float(o) for o in task_outputs_list], dtype=np.float32)
# task_samples[0] already carries delay_type / output_name (validation's CellTestDataset).
task_info = {'delay_type': task_samples[0].get('delay_type', 'rise'),
             'output_name': task_samples[0].get('output_name', '')}
print(f'\ncell={CELL_NAME}  category={CATEGORY}  task_idx={task_idx}/{dataset.num_tasks}')
print(f'  output_name={task_info["output_name"]}  delay_type={task_info["delay_type"]}')
print(f'  y(V) shape={y61_signed.shape}  range=[{y61_signed.min():+.4g}, {y61_signed.max():+.4g}]')
print(f'  signs at V0/V30/V60 = ({np.sign(y61_signed[0]):+.0f}, {np.sign(y61_signed[30]):+.0f}, {np.sign(y61_signed[60]):+.0f})')

## 2. Load model + base norm_stats

In [ ]:
ckpt = torch.load(CKPT_PATH, weights_only=False, map_location=device)
base_norm_stats = ckpt['norm_stats']
print('Checkpoint norm_stats (raw, before alias):')
for k, v in base_norm_stats['node_features'].items():
    mean_str = v.get('mean', '?')
    std_str  = v.get('std', '?')
    print(f'  {k:13s}  mean={mean_str}  std={std_str}')

cfg = ckpt.get('config', {})
node_features = task_samples[0]['node_features'].shape[1]
pooling = cfg.get('pooling', 'mean')
print(f'\nnode_features={node_features}  pooling={pooling}')

def fresh_model():
    m = create_maml_gcn_model(
        node_features=node_features, pooling=pooling, output_dim=1, dropout=0.0,
        conv_hidden_dim=64, num_conv_layers=2, fc_hidden_dim=256, num_fc_layers=2,
    ).to(device)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval()
    return m

## 3. Adaptation — mirror validation/lib-gen exactly

Per-task pipeline:
1. Alias `output_load → input_slew` in norm_stats if constraint.
2. Normalize all 61 lib samples' node_features.
3. v6 sign_flip detection (`y_low` vs `y_high`).
4. Build edge_index from loopclose topology cache (per arc).
5. Compute `center` (0.9V → normalized via mask), get `center_pred`.
6. Get support model predictions → `grad = (y_norm range) / (support_pred range)`.
7. `move = center_pred - y_norm[middle] / grad`.
8. Inner-loop adaptation via `model_functions_at_training_gnn` (selective_adam, 40 steps).
9. Unflip predictions + actuals if `task_sign_flip < 0`.

In [ ]:
def run_adaptation(task_samples, y61_signed, task_info,
                   support_idx=SUPPORT_IDX, is_constraint=IS_CONSTRAINT):
    """Mirror of TSMC_GCN_topology_validation.run_cell_validation per-task block.
    Returns predictions/actuals already unflipped to original sign space."""
    # Step 1: aliased norm_stats for constraint
    ns = copy.deepcopy(base_norm_stats)
    if is_constraint:
        ns['node_features']['output_load'] = dict(ns['node_features']['input_slew'])

    # Step 2: normalize all 61 lib samples
    samples_norm = []
    for s in task_samples:
        s2 = {k: v for k, v in s.items()}
        s2['node_features'] = normalize_node_features(s['node_features'], ns, temp_mode='typical')
        samples_norm.append(s2)

    # Step 3: v6 sign_flip detection
    y_signed = torch.tensor(y61_signed, dtype=torch.float32)
    task_sign_flip = 1.0
    if is_constraint:
        y_low  = y_signed[support_idx[0]].item()
        y_high = y_signed[support_idx[-1]].item()
        if y_high > y_low:
            task_sign_flip = -1.0
            y_signed = -y_signed

    task_outputs_tensor = y_signed
    y_support = task_outputs_tensor[support_idx]
    y_mean = y_support.mean()
    y_std  = y_support.std()
    y_norm = (y_support - y_mean) / y_std

    # Step 4: adjacency / edge_index for this arc
    cell_cache = topology_cache[CELL_NAME]
    out_name = task_info.get('output_name', '')
    delay_type = task_info.get('delay_type', 'rise')
    if 'output_topologies' in cell_cache and out_name in cell_cache['output_topologies']:
        otopo = cell_cache['output_topologies'][out_name]
        side = 'pull_up' if 'rise' in delay_type else 'pull_down'
        adj = otopo[side]['adjacency_matrix']
    else:
        adj = cell_cache.get('adjacency_matrix', torch.zeros((1, 1)))
    edge_index = adj.nonzero().t()

    # Step 5: center voltage (0.9V → normalized via mask, voltage-bearing nodes only)
    center_sample = samples_norm[support_idx[0]]
    center_nf = center_sample['node_features'].clone()
    vs = ns['node_features']['voltage']
    if 'method' in vs and vs['method'] == 'minmax_positive':
        eps = vs.get('epsilon', 0.01)
        norm_nom = eps + (0.9 - vs['min']) / (vs['max'] - vs['min']) * (1 - eps)
    else:
        norm_nom = (0.9 - vs['mean']) / vs['std']
    vmask = center_nf[:, 4] != 0
    center_nf[vmask, 4] = norm_nom

    model = fresh_model()
    with torch.no_grad():
        center_data = Data(x=center_nf, edge_index=edge_index)
        center = model(Batch.from_data_list([center_data]).to(device)).item()

    # Step 6: support predictions for grad scaling
    sup_preds = []
    for idx in support_idx:
        d = Data(x=samples_norm[idx]['node_features'], edge_index=edge_index)
        with torch.no_grad():
            sup_preds.append(model(Batch.from_data_list([d]).to(device)).item())
    sup_preds = torch.tensor(sup_preds)
    min_val, max_val = sup_preds.min().item(), sup_preds.max().item()

    if abs(max_val - min_val) <= 1e-8:
        # Degenerate — flat support output. Fall back to raw passthrough.
        preds = []
        for i in range(61):
            d = Data(x=samples_norm[i]['node_features'], edge_index=edge_index)
            with torch.no_grad():
                preds.append(model(Batch.from_data_list([d]).to(device)).item())
        preds = np.asarray(preds, dtype=float)
        actuals = task_outputs_tensor.numpy().astype(float)
        if task_sign_flip < 0:
            preds   = -preds
            actuals = -actuals
        return dict(predictions=preds, actuals=actuals,
                    support_y=(y_support.numpy() * task_sign_flip),
                    grad=0.0, move=0.0, center=center, adam_used=False,
                    task_sign_flip=task_sign_flip)

    # Step 7: grad / move
    grad = (y_norm.max().item() - y_norm.min().item()) / (max_val - min_val)
    middle_idx = len(support_idx) // 2
    move = center - y_norm[middle_idx].item() / grad

    # Step 8: inner-loop adaptation via gnn_functions.model_functions_at_training_gnn
    y_std1 = y_std * grad
    y_mean1 = y_mean
    y_test = (y_support - y_mean1) / y_std1 + move
    true_function = (task_outputs_tensor - y_mean1) / y_std1 + move

    X_support_samples = [samples_norm[i] for i in support_idx]

    (_m, _o, _l, total_loss, total_mape, predictions, actuals, adam_used, _rmse
    ) = model_functions_at_training_gnn(
        fresh_model(), X_support_samples, y_test.to(device).view(-1, 1),
        true_samples=samples_norm, true_function=true_function,
        topology_cache=topology_cache, cache_type='stage_aware',
        norm_stats=None, normalize_fn=lambda x, *a, **kw: x,  # already normalized
        optim=torch.optim.SGD, lr=0.001, adam_step=40,
        std=y_std1, mean=y_mean1, move=move,
        left_bound=0, right_bound=61, total_points=61, mode='interpolation',
    )
    predictions = np.asarray(predictions, dtype=float)
    actuals     = np.asarray(actuals,     dtype=float)

    # Step 9: unflip predictions/actuals
    if task_sign_flip < 0:
        predictions = -predictions
        actuals     = -actuals

    return dict(predictions=predictions, actuals=actuals,
                support_y=(y_support.numpy() * task_sign_flip),
                grad=float(grad), move=float(move), center=float(center),
                adam_used=bool(adam_used), task_sign_flip=float(task_sign_flip))


result = run_adaptation(task_samples, y61_signed, task_info)

rmse = float(np.sqrt(np.mean((result['predictions'] - result['actuals']) ** 2)))
rng_act = result['actuals'].max() - result['actuals'].min()
nrmse = rmse / max(abs(rng_act), 1e-12) * 100
denom = np.where(np.abs(result['actuals']) > 1e-12, result['actuals'], 1e-12)
mape  = float(np.mean(np.abs((result['predictions'] - result['actuals']) / denom))) * 100
print(f'task_sign_flip = {result["task_sign_flip"]:+.0f}  adam_used = {result["adam_used"]}')
print(f'grad = {result["grad"]:.4g}   move = {result["move"]:.4g}   center = {result["center"]:.4g}')
print(f'NRMSE = {nrmse:.3f}%   MAPE = {mape:.3f}%   RMSE = {rmse:.4g}')

## 4. Plot — actual vs predicted (with support points)

In [ ]:
v_axis = np.arange(61)
fig, ax = plt.subplots(figsize=(9, 4.6))
ax.plot(v_axis, result['actuals'],     '-',  color='#222',    lw=2.0, label='actual (lib)')
ax.plot(v_axis, result['predictions'], '--', color='#1B5E91', lw=1.6, label='predicted')
ax.scatter(SUPPORT_IDX, [result['actuals'][i] for i in SUPPORT_IDX],
           color='#D9534F', edgecolor='black', linewidth=0.5, s=75, zorder=5, label='support (5-shot)')
ax.axhline(0, color='gray', lw=0.6, alpha=0.4)
ax.set_xlabel('voltage index (V_idx → V=0.60..1.20 V)')
ax.set_ylabel(f'{CATEGORY} value (ns, signed)')
title = f'{CELL_NAME} / {CATEGORY} / task {task_idx} — NRMSE={nrmse:.2f}%  RMSE={rmse:.4g} ns'
if result['task_sign_flip'] < 0:
    title += '  (v6 sign_flip ACTIVE)'
ax.set_title(title, fontsize=11)
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Residual + smoothness diagnostic

Two questions:
- Where does the model's prediction diverge from the actual lib curve?
- Is the **actual** lib curve smooth in V, or does it zigzag (lib characterization artifact)?

We count sign-flips in `diff(actual)` to flag zigzag.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)

ax = axes[0]
ax.plot(v_axis, result['predictions'] - result['actuals'], '-', color='#1B5E91', lw=1.5, label='pred − actual')
ax.axhline(0, color='black', lw=0.6, alpha=0.5)
ax.scatter(SUPPORT_IDX, [0]*len(SUPPORT_IDX),
           color='#D9534F', edgecolor='black', linewidth=0.5, s=50, zorder=5, label='support')
ax.set_ylabel('residual (ns)')
ax.set_title('Residual across V')
ax.legend(loc='best'); ax.grid(alpha=0.3)

ax = axes[1]
diffs = np.diff(result['actuals'])
sign_flips = int(np.sum(np.sign(diffs[1:]) * np.sign(diffs[:-1]) < 0))
ax.plot(v_axis[1:], diffs, '-o', color='#222', ms=3.5, lw=1.0, label='Δ actual (consecutive V step)')
ax.axhline(0, color='gray', lw=0.6, alpha=0.4)
ax.set_xlabel('voltage index')
ax.set_ylabel('Δ actual per V step')
ax.set_title(f'Actual curve first-difference — sign-flips = {sign_flips}/{len(diffs)-1}  '
             f'(0 = monotone, large = lib zigzag like 10ps SF)')
ax.legend(loc='best'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'\n>>> sign_flips in actual curve: {sign_flips} / {len(diffs)-1}')
print(f'    max |Δ actual|: {np.abs(diffs).max():.4g} ns')
print(f'    actual range  : {rng_act:.4g} ns')
print(f'    max-step / range = {np.abs(diffs).max()/max(abs(rng_act),1e-12)*100:.1f}%')

## 6. Quick re-roll on 5 random tasks

Spot-check lib zigzag and per-task NRMSE without re-running the full notebook.

In [ ]:
rng2 = random.Random(SEED + 1)
for _ in range(5):
    ti = rng2.sample(range(dataset.num_tasks), 1)[0]
    s, o = dataset.get_all_libs_for_task(ti, clone=True)
    y = np.array([float(v) for v in o], dtype=np.float32)
    inf = {'delay_type': s[0].get('delay_type', 'rise'),
           'output_name': s[0].get('output_name', '')}
    r = run_adaptation(s, y, inf)
    rmse = float(np.sqrt(np.mean((r['predictions'] - r['actuals']) ** 2)))
    rng_act = r['actuals'].max() - r['actuals'].min()
    nrmse = rmse / max(abs(rng_act), 1e-12) * 100
    diffs = np.diff(r['actuals'])
    sf = int(np.sum(np.sign(diffs[1:]) * np.sign(diffs[:-1]) < 0))
    print(f'  task {ti:>5d}  flip={r["task_sign_flip"]:+.0f}  '
          f'adam={int(r["adam_used"])}  '
          f'NRMSE={nrmse:5.2f}%  RMSE={rmse:.4g}  sign-flips={sf}/{len(diffs)-1}')